In [2]:
"""
PROGETTO: Automated Real Estate Valuation System (Boston Dataset)
AUTORE: Massimiliano Izzo
OBIETTIVO: Addestramento di un modello di regressione polinomiale con
           protocollo di validazione e analisi degli scostamenti.
"""
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# 1. CARICAMENTO DATI
url_main = "https://raw.githubusercontent.com/ProfAI/machine-learning-fondamenti/refs/heads/main/datasets/housing.csv"
url_pred = "https://raw.githubusercontent.com/ProfAI/machine-learning-fondamenti/refs/heads/main/datasets/housing_predict.csv"

df = pd.read_csv(url_main, index_col=0)
df_pred_raw = pd.read_csv(url_pred)

# 2. PREPARAZIONE E VALIDAZIONE (TRAIN/TEST SPLIT)
X = df.drop("PRICE", axis=1)
y = df["PRICE"]

# Dividiamo i dati: 80% per l'addestramento, 20% per la validazione
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Creazione caratteristiche polinomiali (Grado 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# 3. TRAINING DEL MODELLO
model = LinearRegression()
model.fit(X_train_poly, y_train)

# 4. ANALISI DELLE PERFORMANCE (SCARTI)
y_test_pred = model.predict(X_test_poly)

mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

# Creazione DataFrame di Validazione per Power BI
# Questo servirà per il grafico "Valore Reale vs Valore Predetto"
df_validation = pd.DataFrame({
    "Valore_Reale": y_test,
    "Valore_Predetto": np.round(y_test_pred, 2),
    "Scostamento_Assoluto": np.round(np.abs(y_test - y_test_pred), 2)
}).reset_index(drop=True)

# 5. PREDIZIONE SUI NUOVI DATI (ANONIMIZZAZIONE)
X_new = df_pred_raw.drop("OWNER", axis=1)
X_new_poly = poly.transform(X_new)
new_predictions = model.predict(X_new_poly)

num_properties = len(df_pred_raw)
anonymous_ids = [f"PROP_{i+1:03d}" for i in range(num_properties)]

df_result = pd.DataFrame({
    "Property_ID": anonymous_ids,
    "Valore_Stimato_K$": np.round(new_predictions, 2)
})

df_final_export = pd.concat([df_result, X_new], axis=1)

# 6. ESPORTAZIONE
# Esportiamo due fogli o due file diversi se necessario.
# Qui salviamo il file principale con le predizioni.
df_final_export.to_excel("Housing_Predictions_Boston_Anonymous.xlsx", index=False)
# Esportiamo anche i dati di validazione per il grafico dello scostamento
df_validation.to_excel("Modello_Validazione_Performance.xlsx", index=False)

print(f"Analisi completata.")
print(f"--- PERFORMANCE MODELLO ---")
print(f"R2 Score (Accuratezza): {r2:.2f}")
print(f"Errore Medio (MAE): {mae:.2f} K$")
print("File pronti per Power BI.")

Analisi completata.
--- PERFORMANCE MODELLO ---
R2 Score (Accuratezza): 0.81
Errore Medio (MAE): 2.58 K$
File pronti per Power BI.
